# 01. Joining the sources

**Goal:** one tidy table with one row per country and year, built from the seven cleaned tables, with before and after row counts at every merge, and a few engineered columns.

**Join key:** `code` (ISO three-letter country code) and `year`. Every cleaned table has both, and each has no duplicate keys (asserted in its own notebook).

**Join type:** outer. A country-year stays in the table if *any* source reports it, so no source's rows are thrown away. Rows with missing values are kept here and handled in the analysis (Phase 4 and 5).

## Which source feeds which variable
OWID and WDI both publish spending, primary and tertiary enrolment and out-of-school. I checked earlier that where they overlap the values are identical (OWID republishes the World Bank and UNESCO numbers). So I pick one source per variable and drop the duplicate columns:

| Variable | Source | Why |
|---|---|---|
| Spending, % of GDP | OWID | Covers the 1970s that my WDI download (from 1980) does not |
| Primary enrolment (gross) | OWID | Same reason |
| Tertiary enrolment (gross) | OWID | Same reason |
| Out-of-school share | OWID | More values (4,693 against 4,268) |
| Total secondary enrolment (gross) | WDI | OWID's file is *upper* secondary only |
| GDP per capita (PPP), persistence to last grade, net enrolment, preprimary enrolment, spending per primary student | WDI | Not available from OWID |
| Income group | World Bank classification file | Not available elsewhere |

The OWID upper-secondary table is not used.

In [7]:
import os
import pandas as pd
import numpy as np

CLEAN = "C:/Users/kevin/capstone_kevina/data/cleaned"
assert os.path.isdir(CLEAN), f"folder not found: {CLEAN}"

In [8]:
# # checks the columns of the cleaned datasets
# for name, df in [("owid_spend", owid_spend), ("owid_prim", owid_prim),
#                  ("owid_ter", owid_ter), ("owid_oos", owid_oos),
#                  ("wdi", wdi), ("income", income)]:
#     print(name, df.columns.tolist())

In [9]:
tables = {
    "owid_spend": ("owid_spend_pct_gdp.csv",       ["entity", "code", "year", "spend_pct_gdp", "suspect"]),
    "owid_prim":  ("owid_enr_primary_gross.csv",   ["entity", "code", "year", "enr_primary_gross"]),
    "owid_ter":   ("owid_enr_tertiary_gross.csv",  ["entity", "code", "year", "enr_tertiary_gross"]),
    "owid_oos":   ("owid_out_of_school_share.csv", ["entity", "code", "year", "out_of_school_share"]),
    "wdi":        ("wdi_clean.csv",                ["code", "country", "year", "gdp_pc_ppp", "enr_secondary_gross",
                                                    "enr_secondary_net", "enr_primary_net", "enr_preprimary",
                                                    "persistence_last_grade", "spend_per_primary_student"]),
    "income":     ("income_groups.csv",            ["code", "year", "income_group"]),
}

loaded = {}
for name, (fname, needed) in tables.items():
    df = pd.read_csv(f"{CLEAN}/{fname}")
    missing = [c for c in needed if c not in df.columns]
    print(f"{name:11} {df.shape!s:12} missing columns: {missing}")
    assert not missing, f"{fname} is missing {missing}. Re-run the cleaning section of that dataset (Restart & Run All)."
    assert df["year"].dtype.kind == "i", f"{fname}: year is not an integer"
    assert not df.duplicated(["code", "year"]).any(), f"{fname}: duplicate code-year keys"
    loaded[name] = df

owid_spend, owid_prim, owid_ter = loaded["owid_spend"], loaded["owid_prim"], loaded["owid_ter"]
owid_oos, wdi_in, income_in = loaded["owid_oos"], loaded["wdi"], loaded["income"]

owid_spend  (5164, 5)    missing columns: []
owid_prim   (7778, 4)    missing columns: []
owid_ter    (5367, 4)    missing columns: []
owid_oos    (4693, 4)    missing columns: []
wdi         (8928, 15)   missing columns: []
income      (8090, 4)    missing columns: []


## 1. Merge one source at a time (before and after row counts)
The measurement sources (OWID and WDI) are joined with an **outer** join on `code` and `year`: the row count can only grow, and the growth is the country-years the new source adds. Income groups are joined with a **left** join. An income classification with no measurement beside it is not an observation, and an outer join would add rows.

In [10]:
k = ["code", "year"]
wdi_keep = ["code", "country", "year", "gdp_pc_ppp", "enr_secondary_gross", "enr_secondary_net",
            "enr_primary_net", "enr_preprimary", "persistence_last_grade", "spend_per_primary_student"]

steps = [
    ("OWID spending (start)", None,    owid_spend.rename(columns={"entity": "country"})[k + ["country", "spend_pct_gdp", "suspect"]]),
    ("+ OWID primary",        "outer", owid_prim[k + ["enr_primary_gross"]]),
    ("+ OWID tertiary",       "outer", owid_ter[k + ["enr_tertiary_gross"]]),
    ("+ OWID out-of-school",  "outer", owid_oos[k + ["out_of_school_share"]]),
    ("+ WDI indicators",      "outer", wdi_in[wdi_keep]),
    ("+ income groups",       "left",  income_in[k + ["income_group"]]),
]

# ---- 1. join, step by step ----
joined, log = None, []
for label, how, right in steps:
    assert not right.duplicated(k).any(), f"duplicate keys in {label}"
    if joined is None:
        joined = right.copy()
        log.append((label, "-", len(right), len(joined)))
        continue
    before = len(joined)
    joined = joined.merge(right, on=k, how=how, suffixes=("", "_wdi"))
    if "country_wdi" in joined.columns:
        joined["country"] = joined["country"].fillna(joined["country_wdi"])
        joined = joined.drop(columns="country_wdi")
    assert not joined.duplicated(k).any(), f"the join created duplicate keys at {label}"
    log.append((label, how, before, len(joined)))

assert "income_group" in joined.columns, joined.columns.tolist()

# one country name per code: OWID name first, then the WDI name
names = (pd.concat([owid_spend[["code", "entity"]], owid_prim[["code", "entity"]],
                    owid_ter[["code", "entity"]], owid_oos[["code", "entity"]],
                    wdi_in[["code", "country"]].rename(columns={"country": "entity"})])
         .drop_duplicates("code").set_index("code")["entity"])
joined["country"] = joined["code"].map(names)

# ---- 2. suspect spending values -> missing (flag kept) ----
joined["spend_suspect"] = joined["suspect"].eq(True)
joined.loc[joined["spend_suspect"], "spend_pct_gdp"] = np.nan
joined = joined.drop(columns="suspect")

# ---- 3. engineered columns ----
n_before = len(joined)

joined["dropout_proxy"] = 100 - joined["persistence_last_grade"]

lag = joined[k + ["spend_pct_gdp"]].copy()
lag["year"] = lag["year"] + 1
lag = lag.rename(columns={"spend_pct_gdp": "spend_lag1"})
assert not lag.duplicated(k).any()
joined = joined.merge(lag, on=k, how="left")

group_median = joined.groupby("income_group")["spend_pct_gdp"].transform("median")
ok = joined["spend_pct_gdp"].notna() & group_median.notna()
joined["high_spend"] = (joined["spend_pct_gdp"] > group_median).astype("Int8").where(ok)

joined["high_enrol_primary"] = (joined["enr_primary_gross"] >= 90).astype("Int8").where(joined["enr_primary_gross"].notna())

assert len(joined) == n_before, "the lag merge changed the row count"
assert not joined.duplicated(k).any()

print(pd.DataFrame(log, columns=["step", "join", "rows before", "rows after"]).to_string(index=False))
print()
print("rows:", f"{len(joined):,}", "| columns:", joined.shape[1], "| countries:", joined['code'].nunique(),
      "| years:", joined['year'].min(), "-", joined['year'].max())
print(joined.columns.tolist())

                 step  join  rows before  rows after
OWID spending (start)     -         5164        5164
       + OWID primary outer         5164        9040
      + OWID tertiary outer         9040        9244
 + OWID out-of-school outer         9244        9245
     + WDI indicators outer         9245       10562
      + income groups  left        10562       10562

rows: 10,562 | columns: 20 | countries: 225 | years: 1970 - 2025
['code', 'year', 'country', 'spend_pct_gdp', 'enr_primary_gross', 'enr_tertiary_gross', 'out_of_school_share', 'gdp_pc_ppp', 'enr_secondary_gross', 'enr_secondary_net', 'enr_primary_net', 'enr_preprimary', 'persistence_last_grade', 'spend_per_primary_student', 'income_group', 'spend_suspect', 'dropout_proxy', 'spend_lag1', 'high_spend', 'high_enrol_primary']


## 2. Checks on the joined table

In [11]:
print("rows               :", f"{len(joined):,}")
print("countries          :", joined["code"].nunique())
print("years              :", joined["year"].min(), "to", joined["year"].max())
print("duplicate keys     :", joined.duplicated(k).sum(), "  (must be 0)")
print("rows with no country name:", joined["country"].isna().sum(), "  (must be 0)")
print()
print("non-missing values per column:")
print(joined.drop(columns=k + ["country"]).notna().sum())

rows               : 10,562
countries          : 225
years              : 1970 to 2025
duplicate keys     : 0   (must be 0)
rows with no country name: 0   (must be 0)

non-missing values per column:
spend_pct_gdp                 5157
enr_primary_gross             7778
enr_tertiary_gross            5367
out_of_school_share           4693
gdp_pc_ppp                    6977
enr_secondary_gross           5358
enr_secondary_net             2517
enr_primary_net               3836
enr_preprimary                5289
persistence_last_grade        3565
spend_per_primary_student     1602
income_group                  7517
spend_suspect                10562
dropout_proxy                 3565
spend_lag1                    5116
high_spend                    4187
high_enrol_primary            7778
dtype: int64


**Row count for the brief.** The joined table is above 10,000 rows. Most rows have gaps in some columns, because the sources cover different countries and years. That is expected and handled in the next step: each analysis uses only the rows where its own columns are filled in.

In [12]:
# missing share per column, so the gaps are documented, not hidden
(joined.isna().mean().mul(100).round(1).sort_values(ascending=False)
       .rename("% missing").to_frame())

,% missing
spend_per_primary_student,84.8
enr_secondary_net,76.2
dropout_proxy,66.2
persistence_last_grade,66.2
enr_primary_net,63.7
high_spend,60.4
out_of_school_share,55.6
spend_lag1,51.6
spend_pct_gdp,51.2
enr_preprimary,49.9


## 3. Missing values and the suspect spending rows

Rows flagged `suspect` in the spending cleaning (below 0.1% or above 25% of GDP: Micronesia 1972, Somalia 2018 to 2023) stay in the table, because they may still have enrollment data. But their spending value is almost certainly wrong, so the spending value is set to missing and the flag is kept in `spend_suspect`. No other values are changed or filled in. I did not impute the missing enrolment or spending values: filling them would invent observations for exactly the variables the question is about.

## 4. Engineered columns

| Column | Definition | Used for |
|---|---|---|
| `dropout_proxy` | `100 - persistence_last_grade` (share of pupils who start primary and do not reach the last grade) | the dropout side of the question |
| `spend_lag1` | the same country's spending **one year earlier** | spending takes time to show up in enrolment |
| `high_spend` | 1 if spending is above the median of its **income group**, else 0 | A/B-style comparison of similar-income countries |
| `high_enrol_primary` | 1 if gross primary enrolment is at least 90%, else 0 | probability and proportion tests |

## 5. How many rows each analysis can use (complete cases)

These are the sample sizes to quote next to every correlation and test.

In [16]:
pairs = {
    "spending vs primary enrolment":              ["spend_pct_gdp", "enr_primary_gross"],
    "  ... plus GDP per capita":                  ["spend_pct_gdp", "enr_primary_gross", "gdp_pc_ppp"],
    "  ... plus income group":                    ["spend_pct_gdp", "enr_primary_gross", "gdp_pc_ppp", "income_group"],
    "spending vs secondary enrolment":            ["spend_pct_gdp", "enr_secondary_gross"],
    "spending vs tertiary enrolment":             ["spend_pct_gdp", "enr_tertiary_gross"],
    "spending vs dropout proxy":                  ["spend_pct_gdp", "dropout_proxy"],
    "spending vs share out of school":            ["spend_pct_gdp", "out_of_school_share"],
    "last year's spending vs primary enrolment":  ["spend_lag1", "enr_primary_gross"],
    "high_spend vs high_enroll_primary":           ["high_spend", "high_enrol_primary"],
}
pd.DataFrame({"rows": {name: len(joined.dropna(subset=cols)) for name, cols in pairs.items()},
              "countries": {name: joined.dropna(subset=cols)["code"].nunique() for name, cols in pairs.items()}})

,rows,countries
spending vs primary enrolment,3900,203
... plus GDP per capita,3206,190
... plus income group,3201,190
spending vs secondary enrolment,3151,196
spending vs tertiary enrolment,3111,195
spending vs dropout proxy,2300,178
spending vs share out of school,2883,198
last year's spending vs primary enrolment,3876,204
high_spend vs high_enroll_primary,3345,199


In [14]:
print("joined  :", joined.columns.tolist())
print("income_in:", income_in.columns.tolist(), income_in.shape)
print(log[-1])          # last join step

joined  : ['code', 'year', 'country', 'spend_pct_gdp', 'enr_primary_gross', 'enr_tertiary_gross', 'out_of_school_share', 'gdp_pc_ppp', 'enr_secondary_gross', 'enr_secondary_net', 'enr_primary_net', 'enr_preprimary', 'persistence_last_grade', 'spend_per_primary_student', 'income_group', 'spend_suspect', 'dropout_proxy', 'spend_lag1', 'high_spend', 'high_enrol_primary']
income_in: ['code', 'economy', 'year', 'income_group'] (8090, 4)
('+ income groups', 'left', 10562, 10562)


## 6. Save

In [17]:
joined.to_csv(f"{CLEAN}/joined.csv", index=False)
print("saved", joined.shape)

saved (10562, 20)
